# **Exercise of Recommender Systems: Implementation of the LightGCN model**

In this Exercise, we will work to construct our This notebook walks through the process of training and evaluating a **LightGCN** model on the Gowalla dataset.

We recommend you save a copy of this colab in your drive so you don't lose progress!

Let's first download the datasets

In [1]:
!git clone https://github.com/huangtinglin/NGCF-PyTorch

Cloning into 'NGCF-PyTorch'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 66 (delta 4), reused 2 (delta 2), pack-reused 55 (from 1)
Receiving objects: 100% (66/66), 11.71 MiB | 25.25 MiB/s, done.
Resolving deltas: 100% (21/21), done.


## **1) Define LightGCN Model**
The LightGCN model consists of:
- User and item embeddings
- Multiple layers of message passing
- A BPR loss function for training

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, num_layers):
        super(LightGCN, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.num_layers = num_layers

        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, user_item_graph):
        all_embeddings = [torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)]

        for _ in range(self.num_layers):
            new_embedding = torch.spmm(user_item_graph, all_embeddings[-1])  # Sparse matrix multiplication
            all_embeddings.append(new_embedding)

        final_embedding = sum(all_embeddings) / (self.num_layers + 1)
        return final_embedding[:self.user_embedding.num_embeddings], final_embedding[self.user_embedding.num_embeddings:]

    def bpr_loss(self, users, pos_items, neg_items, user_item_graph):
        user_emb, item_emb = self.forward(user_item_graph)
        user_vec = user_emb[users]
        pos_vec = item_emb[pos_items]
        neg_vec = item_emb[neg_items]

        pos_scores = (user_vec * pos_vec).sum(dim=1)
        neg_scores = (user_vec * neg_vec).sum(dim=1)

        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()
        return loss

## **3) Load and Process Data**
We load the Gowalla dataset, remap IDs, and construct an adjacency matrix.

In [3]:
import pandas as pd

# Load user ID mappings
user_map = pd.read_csv("NGCF-PyTorch/Data/gowalla/user_list.txt", sep=" ", header=0)
user_id_map = dict(zip(user_map["org_id"], user_map["remap_id"]))

# Load item ID mappings
item_map = pd.read_csv("NGCF-PyTorch/Data/gowalla/item_list.txt", sep=" ", header=0)
item_id_map = dict(zip(item_map["org_id"], item_map["remap_id"]))

print(f"Loaded {len(user_id_map)} user mappings and {len(item_id_map)} item mappings.")

# Load adjacency list from train.txt with remapped IDs
def load_adj_list_with_remap(file_path, user_id_map, item_id_map):
    interactions = []

    with open(file_path, "r") as f:
        for line in f:
            tokens = list(map(int, line.strip().split()))
            if len(tokens) > 1:
                user = tokens[0]
                items = tokens[1:]  # All subsequent values are item IDs

                # Ensure valid remapping
                if user in user_id_map:
                    remapped_user = user_id_map[user]
                    for item in items:
                        if item in item_id_map:
                            remapped_item = item_id_map[item]
                            interactions.append((remapped_user, remapped_item))

    return interactions

# Load remapped adjacency list
train_file = "NGCF-PyTorch/Data/gowalla/train.txt"
gowalla_interactions = load_adj_list_with_remap(train_file, user_id_map, item_id_map)

print(f"Loaded {len(gowalla_interactions)} remapped user-item interactions.")

Loaded 29858 user mappings and 40981 item mappings.
Loaded 46178 remapped user-item interactions.


## **4) Create Adjacency Matrix**
We construct a **symmetric normalized adjacency matrix** for message passing.

In [5]:
import numpy as np
import scipy.sparse as sp
import torch

# Get unique user and item counts
num_users = len(user_id_map)
num_items = len(item_id_map)

def create_adj_matrix(interactions, num_users, num_items):
    """
    Creates a symmetric adjacency matrix for the user-item bipartite graph.

    Args:
        interactions (list of tuples): List of (user, item) interactions.
        num_users (int): Total number of users.
        num_items (int): Total number of items.

    Returns:
        scipy.sparse.csr_matrix: Symmetrically normalized sparse adjacency matrix.
    """
    data = np.ones(len(interactions) * 2)  # Double the data for symmetry
    row = []
    col = []

    for u, i in interactions:
        row.append(u)
        col.append(i + num_users)  # User to item
        row.append(i + num_users)
        col.append(u)  # Item to user (for symmetry)

    # Create sparse adjacency matrix
    adj_matrix = sp.coo_matrix((data, (row, col)), shape=(num_users + num_items, num_users + num_items))

    # Convert to CSR format for efficient operations
    adj_matrix = adj_matrix.tocsr()

    # Symmetric normalization
    rowsum = np.array(adj_matrix.sum(axis=1)).flatten()
    # Ensure no divide-by-zero errors
    rowsum[rowsum == 0] = 1  # Replace zero degrees with 1 to avoid division errors
    d_inv_sqrt = np.where(rowsum > 0, 1.0 / np.sqrt(rowsum), 0)  # Avoid division by zero
    d_mat = sp.diags(d_inv_sqrt)

    # Normalize adjacency matrix: D^{-1/2} * A * D^{-1/2}
    norm_adj_matrix = d_mat @ adj_matrix @ d_mat

    return norm_adj_matrix

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """
    Converts a SciPy sparse matrix to a PyTorch sparse tensor.

    Args:
        sparse_mx (scipy.sparse.csr_matrix): SciPy sparse matrix.

    Returns:
        torch.sparse.FloatTensor: PyTorch sparse tensor.
    """
    sparse_mx = sparse_mx.tocoo().astype(np.float32)  # Convert to COO format
    indices = torch.tensor([sparse_mx.row, sparse_mx.col], dtype=torch.long)
    values = torch.tensor(sparse_mx.data, dtype=torch.float32)
    shape = torch.Size(sparse_mx.shape)

    return torch.sparse.FloatTensor(indices, values, shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generate adjacency matrix
adj_matrix = create_adj_matrix(gowalla_interactions, num_users, num_items)
adj_matrix = sparse_mx_to_torch_sparse_tensor(adj_matrix).to(device)

/tmp/ipython-input-580141375.py:60: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  indices = torch.tensor([sparse_mx.row, sparse_mx.col], dtype=torch.long)
/tmp/ipython-input-580141375.py:64: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:653.)
  return torch.sparse.FloatTensor(indices, values, shape)


In [6]:
from torch.utils.data import DataLoader, Dataset

class BPRDataset(Dataset):
    def __init__(self, interactions, num_users, num_items, num_neg=1):
        self.interactions = interactions
        self.num_users = num_users
        self.num_items = num_items
        self.num_neg = num_neg
        self.user_pos = {u: set() for u in range(num_users)}

        for u, i in interactions:
            self.user_pos[u].add(i)

    def __len__(self):
        return len(self.interactions)

    def __getitem__(self, index):
        user, pos_item = self.interactions[index]

        neg_item = np.random.randint(0, self.num_items)

        # Ensure negative item is truly negative
        while neg_item in self.user_pos[user]:
            neg_item = np.random.randint(0, self.num_items)

        return torch.LongTensor([user]), torch.LongTensor([pos_item]), torch.LongTensor([neg_item])

# Create dataset and dataloader
train_dataset = BPRDataset(gowalla_interactions, num_users, num_items)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)

## **6) Define Training Loop**
We train the model using **Bayesian Personalized Ranking (BPR) loss**.

In [7]:
# Initialize LightGCN
num_layers = 3
embedding_dim = 64

model = LightGCN(num_users, num_items, embedding_dim, num_layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)

# Training function
def train_model(model, train_loader, optimizer, user_item_graph, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            users, pos_items, neg_items = [x.to(device).squeeze() for x in batch]

            optimizer.zero_grad()
            loss = model.bpr_loss(users, pos_items, neg_items, user_item_graph)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Train the model
train_model(model, train_loader, optimizer, adj_matrix, epochs=50)

Epoch 1/50, Loss: 12.0411
Epoch 2/50, Loss: 2.4947
Epoch 3/50, Loss: 0.8514
Epoch 4/50, Loss: 0.5844
Epoch 5/50, Loss: 0.5257
Epoch 6/50, Loss: 0.4644
Epoch 7/50, Loss: 0.4588
Epoch 8/50, Loss: 0.3990
Epoch 9/50, Loss: 0.3975
Epoch 10/50, Loss: 0.3688
Epoch 11/50, Loss: 0.3587
Epoch 12/50, Loss: 0.3131
Epoch 13/50, Loss: 0.3042
Epoch 14/50, Loss: 0.3251
Epoch 15/50, Loss: 0.3018
Epoch 16/50, Loss: 0.2554
Epoch 17/50, Loss: 0.2799
Epoch 18/50, Loss: 0.2672
Epoch 19/50, Loss: 0.2206
Epoch 20/50, Loss: 0.2794
Epoch 21/50, Loss: 0.2260
Epoch 22/50, Loss: 0.2550
Epoch 23/50, Loss: 0.2265
Epoch 24/50, Loss: 0.2308
Epoch 25/50, Loss: 0.2180
Epoch 26/50, Loss: 0.2397
Epoch 27/50, Loss: 0.2371
Epoch 28/50, Loss: 0.1993
Epoch 29/50, Loss: 0.2164
Epoch 30/50, Loss: 0.1903
Epoch 31/50, Loss: 0.2107
Epoch 32/50, Loss: 0.1848
Epoch 33/50, Loss: 0.1765
Epoch 34/50, Loss: 0.1827
Epoch 35/50, Loss: 0.1579
Epoch 36/50, Loss: 0.1801
Epoch 37/50, Loss: 0.1696
Epoch 38/50, Loss: 0.1670
Epoch 39/50, Loss: 0

## **7) Compute Recall@20**
We evaluate model performance using Recall@20.

In [8]:
def load_test_data(file_path, user_map, item_map):
    """
    Loads test interactions and maps raw IDs to LightGCN indices.

    Args:
        file_path (str): Path to the test dataset.
        user_map (dict): Maps raw user IDs to indexed user IDs.
        item_map (dict): Maps raw item IDs to indexed item IDs.

    Returns:
        dict: {mapped_user: set of mapped_item indices}.
    """
    test_data = {}

    with open(file_path, 'r') as f:
        for line in f:
            values = list(map(int, line.strip().split()))
            if len(values) < 2:
                continue  # Skip malformed lines

            raw_user = values[0]
            raw_items = values[1:]  # List of item IDs

            if raw_user in user_map:
                mapped_user = user_map[raw_user]
                mapped_items = {item_map[i] for i in raw_items if i in item_map}  # Map only valid items

                if mapped_items:
                    test_data[mapped_user] = mapped_items

    return test_data

In [9]:
def recall_at_k(true_items, recommended_items, k=20):
    """
    Computes Recall@K for a single user.

    Args:
        true_items (set): The set of ground-truth items the user interacted with.
        recommended_items (list): The top-K recommended items.
        k (int): The cutoff rank.

    Returns:
        float: Recall@K value.
    """
    recommended_items = recommended_items[:k]  # Take top-K recommendations
    hits = len(set(recommended_items) & set(true_items))
    return hits / len(true_items) if true_items else 0

In [10]:
def evaluate_recall(model, test_data, norm_adj_matrix, k=20):
    """
    Computes Recall@K using mapped user-item indices.

    Args:
        model (LightGCN): Trained LightGCN model.
        test_data (dict): {mapped_user: set of mapped_item indices}.
        norm_adj_matrix (torch.sparse.FloatTensor): Normalized adjacency matrix.
        k (int): Cutoff rank.

    Returns:
        float: Mean Recall@K over all test users.
    """
    model.eval()
    user_emb, item_emb = model.forward(norm_adj_matrix)  # Get LightGCN embeddings
    recall_scores = []

    for user, true_items in test_data.items():
        user_vector = user_emb[user]  # Retrieve user embedding
        scores = (user_vector @ item_emb.T).detach().cpu().numpy()  # Compute item scores

        recommended_items = np.argsort(scores)[::-1][:k]  # Get top-K recommendations

        recall_scores.append(recall_at_k(true_items, recommended_items, k))

    return np.mean(recall_scores)  # Average Recall@K

In [11]:
test_data = load_test_data('NGCF-PyTorch/Data/gowalla/test.txt', user_id_map, item_id_map)

# Compute Recall@20
recall_20 = evaluate_recall(model, test_data, adj_matrix, k=20)

print(f"Recall@20: {recall_20:.4f}")

Recall@20: 0.1843
